### **Student Information**
GitHub ID: Gagura1236

Kaggle name: Gagura R

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

# **Project Report: Multi-Architecture Emotion Recognition on Social Media (`v1 -> v17`)**

**Final Private Leaderboard Result — Rank #8 (Macro F1 = `0.6635`):**
* **Selected Slot 1 (`v13 Route A`):** `Submission_v13_RouteA_Top3FT_DisgustCalib.csv` — **Private LB = `0.6635`** | Public LB = `0.6521` | 5-Fold Group-OOF = `0.6581`
* **Selected Slot 2 (`v15 Route B`):** `Submission_v15_RouteB_4ArchSuper_DisgustCalib.csv` — **Private LB = `0.6635`** | Public LB = `0.6434` | 5-Fold Group-OOF = `0.6623`
* Both selected submissions tied for my exact highest Private Leaderboard score (`0.6635`) across all 23 competition entries, with near-zero Group-CV to Private-LB generalization gap (`+0.0012` on `v15 B`, `+0.0054` on `v13 A`).

---

## 1. Preprocessing Steps

Exploratory auditing of `final_posts.json`, `data_identification.csv`, and `emotion.csv` (`3,360` labeled training posts and `1,440` unlabeled test posts across 6 Ekman classes: `joy` 875 / 26.04%, `anger` 674 / 20.06%, `sadness` 628 / 18.69%, `disgust` 454 / 13.51%, `surprise` 441 / 13.12%, and `fear` 288 / 8.57%) surfaced three structural properties that shaped preprocessing:

1. **Dataset Provenance (Reddit GoEmotions + Multilingual/Translated Posts):**
   * Raw posts carry 7-character base-36 Reddit comment IDs (e.g., `f8bv5zh`, `q0rr1l9`) and explicit GoEmotions anonymization placeholders (`[NAME]` in 7.0% of train and 7.7% of test), mixed with multilingual and translated social-media posts (10.2% non-ASCII in train, 11.9% in test, including Nigerian Pidgin, Portuguese-translated, and Emakhuwa/Mozambique posts).
   * **Decision:** Instead of aggressively stripping punctuation, emojis, capitalization, or non-ASCII text (which destroys affective cues like `?!`, `ALL CAPS`, and emojis), I kept raw casing, emojis, punctuation, and `[NAME]` / `@<username>` tokens intact for all Transformer tokenizers, applying only minimal whitespace trimming (`str.strip()`). Classical TF-IDF baselines (`v1-v10`) used sublinear TF scaling with word and character n-grams.

2. **Duplicate & Label-Conflict Handling (Leakage-Free Group Splitting):**
   * Case-insensitive text normalization found **50 duplicate texts within the training split** (**34** with consistent labels and **16** with conflicting emotion labels), along with **59 exact text overlaps** between train and test.
   * **Decision:** Standard `StratifiedKFold` in early runs (`v1-v10`) leaked identical texts across train/validation folds and inflated local CV. From `v12` onward, I enforced **`StratifiedGroupKFold(n_splits=5)` grouped by lowercased normalized text**, ensuring duplicates never cross validation folds. Because 16/50 train duplicates carry conflicting labels, I also dropped test-duplicate hard-locking completely.

3. **Sequence Length Alignment:**
   * Character lengths are closely matched across splits (train mean `87.8`, p95 `181` chars; test mean `84.9`, p95 `167` chars). Setting `max_length=128` covers `>99%` of posts without meaningful truncation while keeping GPU training fast.

---

## 2. Feature Engineering Steps

Feature design progressed through three stages guided by discrepancies between local CV and Kaggle LB:

### Phase 1: Sparse Lexical & General Sentence Embeddings (`v1-v6`, Public LB `0.5393 -> 0.5945`)
* Combined sublinear word (`ngram_range=(1,2)`) and character (`char_wb (3,5)`) TF-IDF features with frozen general-purpose sentence embeddings (`all-MiniLM-L6-v2`, 384d).
* **Bottleneck:** Lexical features and general embeddings struggled on fine-grained affective boundaries, particularly separating `disgust` from `anger` (`disgust` recall stalled around `44.9%`).

### Phase 2: Six Domain-Matched Frozen Large & Multilingual Emotion Backbones (`v11`, Public LB `0.6032 -> 0.6254`, Private LB `0.6453`)
* Guided by the Reddit GoEmotions and multilingual provenance finding, I replaced general encoders with zero-gradient forward extraction (`output_hidden_states=True`, collecting raw emotion logits plus L2-normalized `0.5*mean + 0.5*CLS` penultimate embeddings) across six complementary pretrained models:

| # | Pretrained Backbone | Scale / Hidden | Pretraining Domain & Label Space | Zero-Shot F1 | Frozen 10-Fold OOF F1 |
| :-: | :--- | :---: | :--- | :---: | :---: |
| 1 | `j-hartmann/emotion-english-roberta-large` | 355M / 1024d | 6 emotion datasets, exact 7 Ekman classes | 0.5831 | **0.6235** |
| 2 | `SamLowe/roberta-base-go_emotions` | 125M / 768d | 58,000 Reddit comments, 28 GoEmotions | **0.6145** | **0.6203** |
| 3 | `j-hartmann/emotion-english-distilroberta-base` | 82M / 768d | Exact 7 Ekman classes (distilled) | 0.5714 | **0.6177** |
| 4 | `cardiffnlp/twitter-roberta-large-emotion-latest` | 355M / 1024d | 11 social-media emotions (covers all 6 targets) | 0.5716 | **0.5973** |
| 5 | `AnasAlokla/multilingual_go_emotions` | 178M / 768d | Multilingual 28-class GoEmotions (mBERT) | 0.5694 | **0.5872** |
| 6 | `intfloat/multilingual-e5-large` | 560M / 1024d | Multilingual dense encoder (`query: ` prefix) | — | **0.5790** |

* **Inside-Fold Feature Fusion:** Concatenating the **81 standardized emotion logits** (scaled by `0.20` inside each fold via `StandardScaler`) with **5,376d L2-normalized embeddings** reached `0.6446` OOF from logits alone (`LogisticRegression`, `C=0.02`) and **`0.6562` OOF (`LogLoss = 0.9273`)** across a 5-family ensemble (`Logits-Only LogReg`, `Joint L2-LogReg`, `Calibrated LinearSVC`, `3-Seed Regularized PyTorch MLP`, and `Meta-LogReg`). This produced a `+0.0222` Public LB jump (`0.6032 -> 0.6254`) and `0.6453` Private LB without updating a single Transformer weight.

### Phase 3: Cross-Architecture Representation Fine-Tuning (`v12-v15`)
* To adapt beyond frozen features to the specific annotator boundaries of these 3,360 posts, I trained GPU-accelerated (Colab L4/T4, BF16/FP16 AMP) fine-tuned models across **four distinct Transformer families**: **RoBERTa**, **ModernBERT** (RoPE + GeGLU), **DeBERTa-v3** (disentangled relative attention), and **XLM-RoBERTa** (100-language multilingual).

---

## 3. Model Implementation & Iterative Progression (`v1 -> v17`)

### 3.1 Warm-Start Fine-Tuning Architecture (`v12` & `v14`)
Naive full fine-tuning on 3,360 posts overfits badly (in `v7`, full fine-tuning of Twitter-RoBERTa dropped Public LB from `0.5945` to `0.5604`). Five design choices stabilized fine-tuning:

1. **Warm-Started Classification Heads:**
   * *Exact 6-Class Head Slicing (`v12`)*: For `j-hartmann/emotion-english-roberta-large`, `SamLowe/roberta-base-go_emotions`, `cardiffnlp/twitter-roberta-large-emotion-latest`, and `AnasAlokla/multilingual_go_emotions`, I copied the pretrained weights and biases for the 6 Ekman targets (`[anger, disgust, fear, joy, sadness, surprise]`), starting Epoch 0 around `0.58-0.61` Macro F1 before any gradient step.
   * *Universal `28 -> 6` Emotion Projector (`v14`)*: For the three non-RoBERTa models (`cirimus/modernbert-large-go-emotions`, `darkfordays/deberta-v3-large-go-emotions`, `Prasadrao/xlm-roberta-large-go-emotions-v3`), I retained all 28 pretrained GoEmotions logits and attached a warm-started `nn.Linear(28, 6)` initialized to the identity map on the 6 Ekman indices. This let the model learn auxiliary cues (`annoyance -> anger/disgust`, `disapproval -> disgust`, `nervousness -> fear`, `realization -> surprise`) without cold-start shock.
2. **Selective Lower-Layer Freezing + LLRD:** Froze embeddings and lower blocks (`freeze_below=8-12` for 24-28 layer Large models; `2` for 12-layer Base models) paired with Layer-wise Learning Rate Decay (`llrd=0.92`, `top_lr=1.2e-5-2.5e-5`, `weight_decay=0.02`, cosine schedule with 10% warmup).
3. **Multi-Sample Dropout:** Averaged 5 parallel dropout passes (`p in [0.05, 0.35]`) during training to smooth gradients.
4. **Mild Inverse-Power Class Weighting + Label Smoothing:** Cross-entropy with $w_c \propto (N / N_c)^{0.20}$ (mean-normalized to `1.0`) and `label_smoothing=0.05` to support minority classes (`disgust`, `fear`) without distorting calibration.
5. **Group-CV Early Stopping:** Strict `StratifiedGroupKFold(n_splits=5)` grouped by normalized text, selecting checkpoints strictly on out-of-fold validation Macro F1.

### 3.2 Complete Experimental Progression Table (`v2 -> v17`)

| Version | Route / Architecture | Validation | Local OOF F1 | OOF LogLoss | Public LB (30%) | **Private LB (70%)** | Key Finding |
| :--- | :--- | :--- | :---: | :---: | :---: | :---: | :--- |
| `v2` | Baseline TF-IDF + LinearSVC | 5-Fold SKF | 0.5400 | — | 0.5393 | — | Clean lexical baseline |
| `v6` | LinearSVC + Bagged MLP + MiniLM | 5-Fold SKF | 0.5999 | — | 0.5945 | — | Best classical non-fine-tuned baseline |
| `v7` | Naive Full FT Twitter-RoBERTa | 5-Fold SKF | 0.5995 | — | 0.5604 | — | Severe overfitting (`-0.0391` CV-LB gap) |
| `v8.1 B` | Quad Ensemble (SVC + MLP + 5-Layer FT RoBERTa/DeBERTa) | 5-Fold SKF | 0.6180 | — | 0.6032 | 0.6245 | First clean ensemble break above `0.600` |
| `v9.4 A` | Anti-overfitting RoBERTa/DeBERTa tuning | 5-Fold SKF | — | — | 0.5943 | 0.6078 | Confirmed narrow gains on weak backbones |
| `v10.6 B` | Stacked Heuristics (KNN + Subword + Soft Dup Prior + Specialists) | 10-Fold SKF | 0.6313 | — | 0.6015 | 0.6238 | **Overfitting trap**: scored *below* `v8.1 B` on both Public and Private LB |
| `v11 A` | 6 Frozen Domain Large & Multilingual Backbones (5-Family Pure) | 10-Fold SKF | 0.6562 | 0.9273 | 0.6254 | 0.6453 | `+0.0208` Private LB jump over `v8.1 B` with zero fine-tuning |
| `v12 A` | 4 Colab Warm-Start FT (`Hartmann` 0.6286, `SamLowe` 0.6360, `Cardiff` 0.6183, `mBERT` 0.5929) + 40% `v11` | 5-Fold Group | 0.6564 | — | 0.6369 | 0.6482 | Strong base blend, but `disgust` still under-predicted (8.12%) |
| **`v13 A`** | **Top-3 FT RoBERTa/GoEmotions (40%) + `v11` (60%) + Nested `disgust x 1.20`** | **5-Fold Group** | **0.6581** | **0.9205** | **0.6521** | **`0.6635`** | **Selected Slot 1 (Tied #1 Personal Best Private LB)** |
| `v14 B` | Pure Cross-Arch 3 Titans (`ModernBERT-Large` 0.6383, `XLM-R-Large` 0.6341, `DeBERTa-v3-Large` 0.6295) | 5-Fold Group | 0.6527 | 0.9566 | 0.6397 | 0.6442 | Pure non-RoBERTa track nearly matched `v11 A` on its own |
| **`v15 B`** | **4-Architecture Super-Ensemble (ModernBERT + DeBERTa-v3 + XLM-R + RoBERTa + v11) + `disgust x 1.35`** | **5-Fold Group** | **0.6623** | **0.9126** | **0.6434** | **`0.6635`** | **Selected Slot 2 (Tied #1 Personal Best Private LB; `+0.0201` Public->Private jump)** |
| `v16 A` | `85% v13 + 15% ModernBERT` + Gated Disgust | 5-Fold Group | 0.6620 | 0.9130 | 0.6422 | 0.6556 | Unselected; trailed both chosen slots by `-0.0079` Private |
| `v16 B` | `4-Arch + 50% v11` + Gated Disgust | 5-Fold Group | 0.6616 | 0.9128 | 0.6404 | 0.6612 | Unselected; 3rd best Private score (`-0.0023` behind `v13 A`/`v15 B`) |
| `v17 B` | Per-Model Nested Temperature Scaling (`T_m`) + Convex NLL Hybrid (`disgust x 1.00`) | 5-Fold Group | **0.6642** | 0.9169 | 0.6318 | 0.6577 | Unselected; highest raw Group-OOF, but uncalibrated `disgust` (`x1.00`) lost `-0.0058` Private |

### 3.3 Why Macro-F1 `disgust` Calibration Worked
Under standard `argmax`, well-calibrated models optimize **Accuracy** and systematically under-predict the ambiguous minority class `disgust` (`8.12%-9.10%` predicted on test vs. `13.51%` training prior; OOF Precision `60.6%` vs. Recall `41.6%`) while over-predicting `anger` (`24.6%-25.2%` vs. `20.06%`). Applying a single Group-CV validated multiplier (`disgust x 1.20` in `v13 A`, restoring test `disgust` to `10.35%` / 149 posts; `disgust x 1.35` in `v15 B`) balanced precision and recall directly for Macro F1, lifting Private LB from `0.6482` (`v12 A`) to **`0.6635`** (`+0.0153`).

### 3.4 Post-Competition Private Leaderboard & Shake-Up Validation
After competition close, comparing all 23 submissions on the 70% Private Leaderboard (`1,008` test posts) confirmed four core lessons:

1. **Zero Selection Regret:** My two final selected slots (`v13 Route A` and `v15 Route B`) tied at **`0.6635`**, beating every unselected submission (`v16 B` `0.6612`, `v17 B` `0.6577`, `v16 A` `0.6556`, `v12 A` `0.6482`, `v11 A` `0.6453`, `v14 B` `0.6442`). Rejecting `v17 B` despite its slightly higher local OOF (`0.6642`) saved `0.0058` on Private LB because `v17 B` removed `disgust` calibration (`x1.00`) and down-weighted `SamLowe`.
2. **Near-Zero CV-to-Private Overfitting:** `v15 B`'s 5-fold Group-OOF (`0.6623`) predicted Private LB (`0.6635`) within **`+0.0012`**, and `v13 A` (`0.6581 -> 0.6635`) stayed within **`+0.0054`**. Grouping folds by normalized text eliminated the optimistic leakage seen earlier.
3. **Empirical Falsification of `v10.6` Heuristics:** `v10.6 B` (KNN + subword + soft duplicate priors + specialists) scored only `0.6238` on Private LB — actually **worse** than clean `v8.1 B` (`0.6245`). Discarding heuristics in favor of domain-matched backbones, Group-CV, and cross-architecture fine-tuning added **`+0.0397`** Private Macro F1.
4. **Shake-Up Resilience (Rank #8 Overall):** The 30% Public split (`432` posts) proved `0.011-0.026` harder than the 70% Private split across every model (`v15 B` jumped `+0.0201` from `0.6434` to `0.6635`, catching `v13 A`). Amid large Private Leaderboard swings (+9 to +29 places across ranks 4-7), the dual-track pairing held steady at **Rank #8** (from #7 Public), finishing within `0.0080` of 3rd place (`0.6715`).

---

## 4. AI Prompts and Critical Engineering Decisions (Adopted vs. Rejected)

AI was used as an analytical sounding board for error diagnosis, hypothesis design, and code scaffolding, with every suggestion gated by out-of-fold validation before adoption:

| # | AI Prompt Summary | AI Suggestion | Decision | Quantitative Evidence & Rationale |
| :-: | :--- | :--- | :---: | :--- |
| **1** | *"Analyze why v10.6 achieved 0.6313 local CV yet scored only 0.6015 on Kaggle Public LB, worse than v8.1's 0.6032."* | Audit CV-LB leakage, check raw text provenance, and replace 4-class Twitter-RoBERTa with Reddit GoEmotions + Large Ekman frozen backbones (`v11`). | **Adopted** | Text inspection confirmed Reddit `[NAME]` tokens and multilingual posts; 6 frozen domain backbones raised Public LB `0.6032 -> 0.6254` and Private LB `0.6245 -> 0.6453` (`+0.0208`) with zero fine-tuning overfitting. |
| **2** | *"Evaluate whether frontier LLMs with 15-example domain-calibrated few-shot probability prompting can beat v11 A on sarcastic and translated posts."* | Benchmark `gemini-3-flash-preview`, `gemini-3.1-flash-lite-preview`, `openai/gpt-oss-120b`, and `qwen/qwen3.8-27b` on a stratified 300-post slice before full inference. | **Rejected** | On the exact 300-post slice (`v11 A = 0.6515`), all four LLMs scored `0.5362-0.5637` (`-0.09` to `-0.11` worse), and blending degraded performance (`0.630-0.650`). Rejected LLM prompting and preserved a submission slot. |
| **3** | *"Design a leakage-free fine-tuning pipeline for `SamLowe/roberta-base-go_emotions` and `j-hartmann/emotion-english-roberta-large` that avoids v7's overfitting."* | Use `StratifiedGroupKFold` on normalized text, warm-start 6-class heads from Ekman weights, 5-head Multi-Sample Dropout, LLRD, and mild class weights $(N/N_c)^{0.20}$ on Colab GPU. | **Adopted** | Removed duplicate fold leakage; `SamLowe` hit `0.6360` Group-OOF and `Hartmann-Large` hit `0.6286`, lifting Private LB from `0.6453` (`v11 A`) to **`0.6635`** (`v13 A`). |
| **4** | *"How can we build an orthogonal Route B for Slot 2 to hedge against Private Leaderboard shake-up?"* | Fine-tune 3 non-RoBERTa Large GoEmotions architectures (`ModernBERT-Large`, `DeBERTa-v3-Large`, `XLM-R-Large`) via a universal `28 -> 6` emotion projector. | **Adopted** | `ModernBERT-Large` reached **`0.6383`** (best single model), `XLM-R-Large` `0.6341`, `DeBERTa-v3-Large` `0.6295`. Combining all 4 families dropped OOF LogLoss to `0.9126` and vaulted `v15 B` `+0.0201` from Public (`0.6434`) to tie **`0.6635`** Private LB. |
| **5** | *"Test `disgust x 1.35`, unsupervised Optimal-Transport prior matching, and per-model temperature-calibrated NLL stacking (`v17 B`) to push past `0.6521`."* | Compare `disgust x 1.35` (`v15 B`), Optimal Transport (`v15 A`), and temperature-calibrated raw `argmax` (`v17 B`). | **Partially Rejected** | Retained `v15 B` (tied #1 Private at `0.6635`), but rejected `v17 B` for final slots after seeing it drop Public LB to `0.6318` (confirmed post-hoc: `v17 B` scored `0.6577` Private, `-0.0058` behind our two selected slots). |
| **6** | *"Should we hard-lock test tweets whose normalized text matches training tweets?"* | Hard-lock exact text duplicates (`59` test overlaps) to majority training label (`v10.3-v10.5`). | **Rejected** | 16/50 within-train duplicate texts carried conflicting labels; post-hoc Private LB confirmed `v10.6 B` (`0.6238`) trailed clean `v8.1 B` (`0.6245`). Removed completely from `v11-v17`. |

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
# 1. Preprocessing & data integrity checks (leakage-free group setup)
import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Locate competition data directory cleanly using relative paths
DATA_DIR = "data/competition"
if not os.path.exists(DATA_DIR):
    DATA_DIR = "../Data Mining Class Kaggle Competition/2026-dm-lab-2-private-competition"

LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LABEL2ID = {lbl: idx for idx, lbl in enumerate(LABELS)}

def load_competition_data(data_dir):
    """Load posts, enforce exact ID alignment, and build normalized-text groups to prevent CV leakage."""
    with open(os.path.join(data_dir, "final_posts.json"), "r", encoding="utf-8") as f:
        post_map = {p["post_id"]: str(p["text"]).strip() for p in json.load(f)}

    df_id = pd.read_csv(os.path.join(data_dir, "data_identification.csv"))
    df_emo = pd.read_csv(os.path.join(data_dir, "emotion.csv"))

    train_ids = df_id[df_id["set"] == "train"]["post_id"].values
    test_ids = df_id[df_id["set"] == "test"]["post_id"].values
    assert (train_ids == df_emo["post_id"].values).all(), "Train ID alignment mismatch!"

    # Preserve raw emojis, casing, punctuation, and [NAME] tokens for pretrained Transformers
    train_texts = [post_map[pid] for pid in train_ids]
    test_texts = [post_map[pid] for pid in test_ids]
    y_train = df_emo["label"].map(LABEL2ID).values

    # Group by lowercased normalized text so duplicate posts never cross train/val folds
    groups = np.array([t.strip().lower() for t in train_texts])
    return train_ids, train_texts, y_train, groups, test_ids, test_texts

train_ids, train_texts, y_train, groups, test_ids, test_texts = load_competition_data(DATA_DIR)

# Data integrity & provenance audit
tr_series = pd.Series(train_texts)
te_series = pd.Series(test_texts)
print(f"Loaded {len(train_texts)} training posts and {len(test_texts)} test posts across {len(LABELS)} classes.")
print(f"Train class distribution: { {LABELS[i]: int((y_train == i).sum()) for i in range(6)} }")
name_pat = r"\[NAME\]"
tr_name_pct = tr_series.str.contains(name_pat).mean() * 100
te_name_pct = te_series.str.contains(name_pat).mean() * 100
print(f"Provenance signals -> [NAME] token: train={tr_name_pct:.1f}%, test={te_name_pct:.1f}%")
print(f"Non-ASCII / multilingual posts: train={tr_series.apply(lambda s: any(ord(c)>127 for c in s)).mean()*100:.1f}%, test={te_series.apply(lambda s: any(ord(c)>127 for c in s)).mean()*100:.1f}%")
print(f"Within-train duplicate normalized texts: {pd.Series(groups).duplicated().sum()} (protected via StratifiedGroupKFold)")

Loaded 3360 training posts and 1440 test posts across 6 classes.
Train class distribution: {'anger': 674, 'disgust': 454, 'fear': 288, 'joy': 875, 'sadness': 628, 'surprise': 441}
Provenance signals -> [NAME] token: train=7.0%, test=7.7%
Non-ASCII / multilingual posts: train=10.2%, test=11.9%
Within-train duplicate normalized texts: 50 (protected via StratifiedGroupKFold)


## 2. Feature Engineering Steps

In [2]:
# 2. Feature engineering: six domain-matched frozen large & multilingual backbones (v11)
import gc
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

# Six complementary domain-pretrained backbones discovered via dataset provenance analysis
FROZEN_CLF_BACKBONES = [
    ("j-hartmann/emotion-english-roberta-large", "hartmann_roberta_large", 32),
    ("cardiffnlp/twitter-roberta-large-emotion-latest", "cardiff_twitter_roberta_large_emo", 32),
    ("SamLowe/roberta-base-go_emotions", "samlowe_goemotions_base", 64),
    ("AnasAlokla/multilingual_go_emotions", "multilingual_goemotions", 64),
    ("j-hartmann/emotion-english-distilroberta-base", "hartmann_distilroberta", 64),
]
FROZEN_EMB_BACKBONE = ("intfloat/multilingual-e5-large", "multilingual_e5_large", "query: ", 32)

@torch.no_grad()
def extract_frozen_emotion_features(model_name, texts, batch_size=32, max_length=160):
    """Extract raw emotion logits + 0.5*(mean + CLS) penultimate embeddings with zero gradient updates."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE)
    model.eval()

    logits_list, embs_list = [], []
    for start in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[start:start + batch_size],
            padding=True, truncation=True, max_length=max_length, return_tensors="pt"
        ).to(DEVICE)
        out = model(**enc, output_hidden_states=True)
        last_hidden = out.hidden_states[-1]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        mean_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        cls_emb = last_hidden[:, 0, :]
        pooled = (0.5 * mean_emb + 0.5 * cls_emb).detach().cpu().float().numpy()

        logits_list.append(out.logits.detach().cpu().float().numpy())
        embs_list.append(pooled)

    del model, tokenizer
    gc.collect()
    return np.vstack(logits_list).astype(np.float32), normalize(np.vstack(embs_list).astype(np.float32))

def fit_inside_fold_linear_probe(X_tr_blocks, X_te_blocks, y, scale_factors, C=0.1, n_splits=10):
    """Strictly inside-fold StandardScaler + L2-regularized LogisticRegression (zero feature leakage)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.zeros((len(y), len(LABELS)), dtype=np.float32)
    test_probs = np.zeros((X_te_blocks[0].shape[0], len(LABELS)), dtype=np.float32)

    for tr, va in skf.split(np.zeros(len(y)), y):
        tr_parts, va_parts, te_parts = [], [], []
        for Xtr, Xte, sf in zip(X_tr_blocks, X_te_blocks, scale_factors):
            if sf is not None:
                sc = StandardScaler()
                tr_parts.append(sc.fit_transform(Xtr[tr]) * sf)
                va_parts.append(sc.transform(Xtr[va]) * sf)
                te_parts.append(sc.transform(Xte) * sf)
            else:
                tr_parts.append(Xtr[tr]); va_parts.append(Xtr[va]); te_parts.append(Xte)

        clf = LogisticRegression(C=C, max_iter=1500, random_state=42)
        clf.fit(np.hstack(tr_parts), y[tr])
        oof[va] = clf.predict_proba(np.hstack(va_parts))
        test_probs += clf.predict_proba(np.hstack(te_parts)) / n_splits

    return oof, test_probs, f1_score(y, np.argmax(oof, axis=1), average="macro")

print(f"Feature extraction pipeline configured for {len(FROZEN_CLF_BACKBONES) + 1} frozen backbones on {DEVICE}.")

Feature extraction pipeline configured for 6 frozen backbones on mps.


## 3. Model Implementation Steps

In [3]:
# 3. Model implementation: warm-started cross-architecture fine-tuning (v12 & v14)
import re
import copy
import math
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup
from scipy.special import softmax

LABEL_ALIASES = {
    "anger": ["anger"], "disgust": ["disgust"], "fear": ["fear"],
    "joy": ["joy", "happy", "happiness"], "sadness": ["sadness", "sad"], "surprise": ["surprise"],
}

class TextDataset(Dataset):
    """Tokenized dataset with max_length=128 covering >99% of posts."""
    def __init__(self, texts, tokenizer, labels=None, max_length=128):
        self.enc = tokenizer(texts, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt")
        self.labels = labels

    def __len__(self):
        return self.enc["input_ids"].shape[0]

    def __getitem__(self, idx):
        item = {"input_ids": self.enc["input_ids"][idx], "attention_mask": self.enc["attention_mask"][idx]}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def extract_layer_index(param_name):
    """Architecture-agnostic layer parser for RoBERTa, ModernBERT, DeBERTa-v3, and XLM-RoBERTa."""
    match = re.search(r"\.layers?\.(\d+)\.", param_name)
    return int(match.group(1)) if match else None

class UniversalEmotionProjectorModel(nn.Module):
    """
    Universal Warm-Started Emotion Classifier (used for ModernBERT-Large, DeBERTa-v3-Large, XLM-R-Large).
    Retains all 28 pretrained GoEmotions output logits and appends a warm-started Linear(28 -> 6)
    projection initialized to the exact Ekman identity mapping, regularized by 5-head Multi-Sample Dropout.
    """
    def __init__(self, model_name, freeze_below=12, dropout_rates=(0.05, 0.10, 0.15, 0.20, 0.25)):
        super().__init__()
        self.backbone = AutoModelForSequenceClassification.from_pretrained(model_name)
        inv = {str(v).lower(): int(k) for k, v in self.backbone.config.id2label.items()}

        # Warm-start 28 -> 6 projection so Epoch 0 starts directly at Zero-Shot performance
        self.proj = nn.Linear(len(inv), len(LABELS))
        with torch.no_grad():
            self.proj.weight.zero_()
            self.proj.bias.zero_()
            for out_i, lbl in enumerate(LABELS):
                src_i = next(inv[a] for a in LABEL_ALIASES[lbl] if a in inv)
                self.proj.weight[out_i, src_i] = 1.0

        # Freeze embeddings and lower transformer blocks to prevent small-sample overfitting
        for name, param in self.backbone.named_parameters():
            if "embeddings" in name or "tok_embeddings" in name:
                if freeze_below > 0:
                    param.requires_grad = False
            else:
                li = extract_layer_index(name)
                if li is not None and li < freeze_below:
                    param.requires_grad = False

        self.dropouts = nn.ModuleList([nn.Dropout(r) for r in dropout_rates])

    def forward(self, input_ids, attention_mask):
        raw_logits = self.backbone(input_ids=input_ids, attention_mask=attention_mask).logits
        if self.training:
            return sum(self.proj(dp(raw_logits)) for dp in self.dropouts) / len(self.dropouts)
        return self.proj(raw_logits)

def build_llrd_optimizer_groups(model, top_lr=1.5e-5, head_lr=6e-5, proj_lr=5e-3, llrd=0.92, weight_decay=0.02):
    """Construct Layer-wise Learning Rate Decay (LLRD) parameter groups for AdamW."""
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias", "norm.weight")
    layer_ids = [extract_layer_index(n) for n, _ in model.backbone.named_parameters()]
    max_layer = max([i for i in layer_ids if i is not None], default=23)

    buckets = {}
    for name, param in model.backbone.named_parameters():
        if not param.requires_grad:
            continue
        wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        li = extract_layer_index(name)
        if li is not None:
            lr = top_lr * (llrd ** (max_layer - li))
        elif "embeddings" in name or "tok_embeddings" in name:
            lr = top_lr * (llrd ** (max_layer + 2))
        else:
            lr = head_lr
        buckets.setdefault((round(lr, 10), wd), []).append(param)

    groups = [{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in buckets.items()]
    groups.append({"params": [model.proj.weight], "lr": proj_lr, "weight_decay": 0.01})
    groups.append({"params": [model.proj.bias], "lr": proj_lr, "weight_decay": 0.0})
    return groups

# Complete 7 fine-tuned models trained via 5-Fold StratifiedGroupKFold on Colab L4/T4 GPU (BF16/FP16 AMP):
#   Route A Family (v12):
#     1. j-hartmann/emotion-english-roberta-large  (355M, freeze_below=8)  -> Group-OOF = 0.6286
#     2. SamLowe/roberta-base-go_emotions          (125M, freeze_below=2)  -> Group-OOF = 0.6360
#     3. cardiffnlp/twitter-roberta-large-emotion-latest (355M, freeze=8)  -> Group-OOF = 0.6183
#     4. AnasAlokla/multilingual_go_emotions       (178M mBERT, freeze=2)  -> Group-OOF = 0.5929
#   Route B Cross-Architecture Titans (v14):
#     5. cirimus/modernbert-large-go-emotions      (395M ModernBERT-Large) -> Group-OOF = 0.6383 (Best single model)
#     6. Prasadrao/xlm-roberta-large-go-emotions-v3(560M XLM-R-Large)      -> Group-OOF = 0.6341
#     7. darkfordays/deberta-v3-large-go-emotions  (435M DeBERTa-v3-Large) -> Group-OOF = 0.6295
print("Warm-start Cross-Architecture Fine-Tuning module defined successfully.")

Warm-start Cross-Architecture Fine-Tuning module defined successfully.


## 4. Cross-Architecture Ensembling, Calibration & Final Verification

In [4]:
# 4. Cross-architecture ensembling, Macro-F1 calibration, and final submission verification
from sklearn.metrics import classification_report

def evaluate_final_dual_track(data_dir, y_true, test_ids):
    """
    Reproduce and verify both selected Kaggle Final Submissions from saved 5-Fold Group-CV probabilities:
      - Slot 1 (v13 Route A): 0.60*v11_frozen + 0.18*Hartmann + 0.18*SamLowe + 0.04*Cardiff + disgust x 1.20
      - Slot 2 (v15 Route B): 4-Architecture Super-Ensemble (ModernBERT + DeBERTa-v3 + XLM-R + RoBERTa + v11) + disgust x 1.35
    """
    cache_dir = data_dir if os.path.exists(os.path.join(data_dir, "v14_routeB_probs.npz")) else "../Data Mining Class Kaggle Competition/2026-dm-lab-2-private-competition"
    p_v12 = os.path.join(cache_dir, "v12_colab_probs.npz")
    p_v14 = os.path.join(cache_dir, "v14_routeB_probs.npz")
    p_v11 = os.path.join(cache_dir, "v11_best_pure_probs.npz")

    if not (os.path.exists(p_v11) and os.path.exists(p_v12) and os.path.exists(p_v14)):
        print("Cached GPU probability matrices not found locally; run Sections 2-3 to regenerate.")
        return

    d11, d12, d14 = np.load(p_v11), np.load(p_v12), np.load(p_v14)

    # Slot 1: v13 Route A (Public LB = 0.6521, Private LB = 0.6635)
    oof_v13 = 0.60 * d11["oof"] + 0.18 * d12["oof_hl"] + 0.18 * d12["oof_go"] + 0.04 * d12["oof_cl"]
    te_v13  = 0.60 * d11["test"] + 0.18 * d12["te_hl"]  + 0.18 * d12["te_go"]  + 0.04 * d12["te_cl"]
    m_v13   = np.array([1.0, 1.20, 1.0, 1.0, 1.0, 1.0])
    oof_p13 = np.argmax(oof_v13 * m_v13, axis=1)
    te_p13  = np.argmax(te_v13  * m_v13, axis=1)

    # Slot 2: v15 Route B — 4-Architecture Super-Ensemble (Public LB = 0.6434, Private LB = 0.6635)
    oof_v15 = d14["oof_grand"]  # 35% Cross-Arch (ModernBERT+DeBERTa-v3+XLM-R) + 65% Route A
    te_v15  = d14["te_grand"]
    m_v15   = np.array([1.0, 1.35, 1.0, 1.0, 1.0, 1.0])
    oof_p15 = np.argmax(oof_v15 * m_v15, axis=1)
    te_p15  = np.argmax(te_v15  * m_v15, axis=1)

    ll_13 = -float(np.mean(np.log(np.clip(oof_v13[np.arange(len(y_true)), y_true], 1e-6, 1.0))))
    ll_15 = -float(np.mean(np.log(np.clip(oof_v15[np.arange(len(y_true)), y_true], 1e-6, 1.0))))

    print("-" * 92)
    print(f"Slot 1 (v13 Route A) | Group-OOF F1 = {f1_score(y_true, oof_p13, average='macro'):.4f} | LogLoss = {ll_13:.4f} | Public = 0.6521 | Private = 0.6635")
    print(f"Slot 2 (v15 Route B) | Group-OOF F1 = {f1_score(y_true, oof_p15, average='macro'):.4f} | LogLoss = {ll_15:.4f} | Public = 0.6434 | Private = 0.6635")
    print("-" * 92)
    print("\nDetailed OOF Classification Report — Slot 2 (4-Architecture Super-Ensemble):")
    print(classification_report(y_true, oof_p15, target_names=LABELS, digits=4))

    # Verify final test prediction counts (1,440 predictions, 0 NaNs)
    for tag, te_preds in [("Slot 1 (v13 Route A)", te_p13), ("Slot 2 (v15 Route B)", te_p15)]:
        dist = {LABELS[c]: int((te_preds == c).sum()) for c in range(6)}
        print(f"{tag} test distribution (n={len(te_preds)}): {dist}")

evaluate_final_dual_track(DATA_DIR, y_train, test_ids)

--------------------------------------------------------------------------------------------
Slot 1 (v13 Route A) | Group-OOF F1 = 0.6577 | LogLoss = 0.9205 | Public = 0.6521 | Private = 0.6635
Slot 2 (v15 Route B) | Group-OOF F1 = 0.6623 | LogLoss = 0.9127 | Public = 0.6434 | Private = 0.6635
--------------------------------------------------------------------------------------------

Detailed OOF Classification Report — Slot 2 (4-Architecture Super-Ensemble):
              precision    recall  f1-score   support

       anger     0.6280    0.6513    0.6395       674
     disgust     0.5116    0.4846    0.4977       454
        fear     0.7705    0.6528    0.7068       288
         joy     0.7566    0.8171    0.7857       875
     sadness     0.6837    0.7022    0.6929       628
    surprise     0.6877    0.6190    0.6516       441

    accuracy                         0.6774      3360
   macro avg     0.6730    0.6545    0.6623      3360
weighted avg     0.6762    0.6774    0.6757   